In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm joblib

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


In [8]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv(
    url,
    names=columns,
    na_values="?",
    skipinitialspace=True
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (32561, 15)


In [9]:
df.head()
df.info()
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_st

In [12]:
df["income"] = df["income"].str.strip()

df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

print(df["income"].value_counts())

income
0    24720
1     7841
Name: count, dtype: int64


In [14]:
df = df.dropna().reset_index(drop=True)

print("Shape after removing missing values:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sum())

Shape after removing missing values: (30162, 15)

Missing values:
0


In [16]:
X = df.drop("income", axis=1)
y = df["income"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (30162, 14)
y shape: (30162,)

Target distribution:
income
0    22654
1     7508
Name: count, dtype: int64


In [18]:
X = pd.get_dummies(X, drop_first=True)

print("Encoded X shape:", X.shape)
print("\nData types:")
print(X.dtypes.value_counts())

Encoded X shape: (30162, 96)

Data types:
bool     90
int64     6
Name: count, dtype: int64


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (24129, 96)
Testing data: (6033, 96)

Training target distribution:
income
0    0.751088
1    0.248912
Name: proportion, dtype: float64

Testing target distribution:
income
0    0.751036
1    0.248964
Name: proportion, dtype: float64


In [22]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [24]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [26]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [28]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=-1,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)
lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]

print("LightGBM trained successfully.")

LightGBM trained successfully.


In [30]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [32]:
baseline_results = [
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_pred,
        logistic_prob
    ),

    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_prob
    ),

    evaluate_model(
        "XGBoost",
        y_test,
        xgb_pred,
        xgb_prob
    ),

    evaluate_model(
        "LightGBM",
        y_test,
        lgbm_pred,
        lgbm_prob
    )
]

baseline_df = pd.DataFrame(baseline_results)

baseline_df = baseline_df.sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

baseline_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,0.865241,0.770621,0.653129,0.707027,0.925456
1,LightGBM,0.865573,0.770133,0.655792,0.708378,0.922963
2,Random Forest,0.848003,0.727981,0.621838,0.670736,0.903497
3,Logistic Regression,0.847340,0.734085,0.606525,0.664236,0.902301


In [34]:
## Hyperparameter Optimization

Hyperparameter optimization is performed using GridSearchCV with 5-fold
Stratified Cross-Validation.

F1-Score is used as the optimization metric because it balances precision
and recall for the classification problem.

SyntaxError: invalid syntax (4256144224.py, line 3)

In [36]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified Cross-Validation created successfully.")

5-Fold Stratified Cross-Validation created successfully.


In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

logistic_grid.fit(X_train, y_train)

print("Best Logistic Regression Parameters:")
print(logistic_grid.best_params_)

print("\nBest Cross-Validation F1-Score:")
print(round(logistic_grid.best_score_, 4))

In [ ]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)

print("\nBest Cross-Validation F1-Score:")
print(round(rf_grid.best_score_, 4))

In [ ]:
lgbm_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 10, 20]
}

lgbm_grid = GridSearchCV(
    estimator=LGBMClassifier(
        random_state=42,
        verbose=-1
    ),
    param_grid=lgbm_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

lgbm_grid.fit(X_train, y_train)

print("Best LightGBM Parameters:")
print(lgbm_grid.best_params_)

print("\nBest Cross-Validation F1-Score:")
print(round(lgbm_grid.best_score_, 4))